# Lab 08｜只在需要时，把下一层资料交给模型

一项 Skill 可以引用很长的运行手册，但没有必要在每次对话开始时全部装入。我们用三个不同的随机标记，分别追踪描述、SKILL.md 正文和参考文件，直接检查它们何时进入模型请求。
本例显式使用 `SkillInput`，不评估自动匹配 Skill 的选择质量。先观察描述、正文和参考文件三层加载，再增加一个极小的脚本执行例子，比较源码和输出是否进入 Context。

## 核心代码：从能力描述到正文，再按任务需要读取参考

> 本讲重点看下面这 6 行。它们摘自后面的代码单元格，仅供定位；运行时仍从公共准备开始按顺序执行。单元格从标题所在的第 1 格起计数，行号按格内源码计算。

**第 1 节，第 5 个单元格，第 2 行：**

```python
greeting = run(thread,"请只回复准备就绪。不要调用任何 Skill 或工具。")
```

普通问候不显式启用 Skill，用来观察能力描述进入输入时，正文与参考内容是否仍未加载。

**第 2 节，第 7 个单元格，第 2 至 3 行：**

```python
method = run(thread,[SkillInput(name="pool-check",path=str(skill_path)),
    TextInput("按这项 Skill 简述核对方法，以及需要阈值时应读哪份文件。此时不需要阈值，不调用工具。")])
```

`SkillInput` 显式加载正文，当前任务还不需要阈值，因此暂不读取参考文件。

**第 3 节，第 9 个单元格，第 2 至 4 行：**

```python
decision = run(thread,
    "现在按教学阈值核对：最近 5 分钟连接占用 95%，连接等待 1200ms。"
    "请读取 Skill 指定的参考文件，判断是否需要进一步检查。不要修改文件。")
```

任务开始需要阈值时，再请 Agent 读取 Skill 指向的参考文件。这 3 个阶段的实际加载情况由后面的请求对照表展示。


## 公共准备｜必跑，代码可折叠

选择 **Python 3.12** 内核，保留仓库的 `examples/notebook_support.py`。下格会自动补装固定版本的实验依赖；如果当前内核已经导入其他版本，请安装后重启内核。无需先到 README 手动安装。

九讲统一读取 `GLM_API_KEY`；未设置时弹出隐藏输入框。模型由 GLM-5.2 提供，不需要 OpenAI / ChatGPT 账号；开始任务后会产生 API 费用。准备区只启动本地连接，不调用模型。每讲可从本格独立启动，最后一格关闭连接；再次从头运行前先执行清理。

版本检查、协议适配、健康检查和退出处理属于辅助代码，不必逐行阅读。主流程以“必读必跑”标注；“扩展/可选”默认跳过。


In [ ]:
import sys, os, json, secrets, time, subprocess, getpass
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
from IPython.display import display, Markdown, HTML
from html import escape
from contextlib import redirect_stdout
from io import StringIO

if sys.version_info[:2] != (3, 12):
    raise RuntimeError("请选择 Python 3.12 内核，再从头运行。")
required = {"openai-codex": "0.154.0", "litellm": "1.101.0"}
missing = []
for package, expected in required.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    if installed != expected:
        missing.append(("litellm[proxy]" if package == "litellm" else package) + "==" + expected)
if missing:
    already_loaded = any(name in sys.modules for name in ("openai_codex", "litellm"))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    if already_loaded:
        raise RuntimeError("依赖已安装；请重启内核并重新运行，避免混用旧模块。")

here = Path.cwd().resolve()
examples = next((p for base in (here, *here.parents)
                 for p in (base, base / "examples") if (p / "notebook_support.py").is_file()), None)
if examples is None:
    raise FileNotFoundError("请在完整仓库内打开本 notebook。")
sys.path.insert(0, str(examples))
from notebook_support import Lab, command_receipts, tool_output_contains
from openai_codex import ApprovalMode, Sandbox
if not os.environ.get("GLM_API_KEY", "").strip():
    os.environ["GLM_API_KEY"] = getpass.getpass("GLM_API_KEY：").strip()
if "lab" in globals() and not lab._closed:
    raise RuntimeError("本讲连接仍在运行，请先执行最后的 lab.close()。")
lab = Lab(examples / "08-progressive-disclosure")

# 展示函数只整理真实材料，不把预期结果冒充本次运行结果。
def table(headers, rows):
    head = "".join(f"<th>{escape(str(v))}</th>" for v in headers)
    body = "".join("<tr>" + "".join(
        "<td><pre style='white-space:pre-wrap;max-width:52em'>" + escape(lab.clean(v)) + "</pre></td>"
        for v in row) + "</tr>" for row in rows)
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))

def details(title, value):
    display(HTML("<details><summary>" + escape(title) + "</summary><pre style='white-space:pre-wrap'>"
                 + escape(lab.clean(value)) + "</pre></details>"))

def run(thread, prompt, **kwargs):
    # 保留运行错误；仅收起辅助层重复打印的状态和回答。
    with redirect_stdout(StringIO()):
        return lab.run(thread, prompt, **kwargs)

def events(result):
    rows = []
    for wrapped in result.items:
        item = wrapped.root
        if item.type == "commandExecution":
            rows.extend([("采取的动作", item.command),
                         ("刚获得的信息", item.aggregated_output or "（无文本输出）")])
        elif item.type == "fileChange":
            rows.append(("文件操作", ", ".join(change.path for change in item.changes)))
        elif item.type == "agentMessage" and item.text.strip():
            rows.append(("Agent 的判断", item.text))
    table(["事件", "本次实际内容"], rows)

def snapshot(folder):
    return {str(p.relative_to(folder)):p.read_bytes() for p in folder.rglob("*") if p.is_file()}

def compare_inputs(folder, before):
    after = snapshot(folder)
    changed = [name for name in sorted(before.keys() | after.keys()) if before.get(name) != after.get(name)]
    print("输入文件：" + ("有变化，展开下方查看" if changed else "未改变"))
    for name in changed:
        details(name, "运行前：\n" + before.get(name,"[不存在]".encode()).decode(errors="replace")
                + "\n运行后：\n" + after.get(name,"[不存在]".encode()).decode(errors="replace"))


WORK = lab.work
import secrets
from openai_codex import SkillInput, TextInput
meta_tag = "META-" + secrets.token_hex(5)
body_tag = "BODY-" + secrets.token_hex(5)
ref_tag = "REF-" + secrets.token_hex(5)
skill_dir = WORK / ".agents" / "skills" / "pool-check"
(skill_dir / "references").mkdir(parents=True)
skill_path = skill_dir / "SKILL.md"
skill_path.write_text(f"""---
name: pool-check
description: 排查连接池等待时使用；普通问候不使用。能力目录标记 {meta_tag}。
---
# 连接池等待核对
方法正文标记：{body_tag}。
先区分连接占用、连接等待与下游耗时，不因等待升高就断言数据库故障。
只有任务需要具体阈值和采样窗口时，读取 references/window.md，再用其中的条件判断。
本方法只做只读核对，不修改连接池配置。
""", encoding="utf-8")
(skill_dir / "references" / "window.md").write_text(
    f"参考标记：{ref_tag}\n教学核对条件：同一 5 分钟窗口内，连接占用超过 90%，且连接等待高于 800ms，才进一步核对池配置与慢查询。\n",
    encoding="utf-8")

script_tag = "SOURCE-" + secrets.token_hex(5)
output_tag = "RESULT-" + secrets.token_hex(5)
(skill_dir / "scripts").mkdir()
script_path = skill_dir / "scripts" / "check_report.py"
script_path.write_text(
    "# " + script_tag + "\nfrom pathlib import Path\nimport sys\n"
    "text = Path(sys.argv[1]).read_text(encoding='utf-8')\n"
    "print('报告字符数：', len(text))\n"
    + "print(" + repr(output_tag) + ")\n", encoding="utf-8")
# 报告属于本次任务，放在 Agent 工作目录；绝对路径不受执行时切换目录影响。
report_path = WORK / "report.md"
report_path.write_text("""# 连接池等待升高事故调查报告（模拟）

## 现象与采样窗口
09:00—09:05，连接占用为 95%，连接等待为 1200ms，部分请求响应变慢。

## 初步判断
同一 5 分钟窗口内，两项指标均超过教学核对阈值，需要进一步检查。
目前尚不能仅凭连接等待升高断言数据库故障。

## 后续核对
核对连接池配置、慢查询和下游耗时，区分连接占用与等待的原因。
本次只记录观察与调查计划，未修改连接池配置。
""", encoding="utf-8")
with skill_path.open("a", encoding="utf-8") as handle:
    handle.write(
        "\n需要检查报告长度时，执行相对于本 SKILL.md 所在目录的 scripts/check_report.py。"
        "第一个参数是待检查报告的路径；报告位于 Agent 工作目录，不在 Skill 目录。"
        f'本次执行命令：`python3 "{script_path}" "{report_path}"`。'
        "使用绝对路径，正常执行不必读取脚本源码。\n")
thread = lab.thread()

print("公共准备完成。")


## 1. 普通问候：先看能力目录｜必读必跑

已准备好 Skill 的名称描述、`SKILL.md` 正文、`references/window.md` 和一个短脚本，各有内部标记。先进行普通问候：名称描述可以进入输入，正文与参考内容是否也进入，最后集中看表。我们查真实请求，不靠模型自述。


In [ ]:
catalog_start = len(lab.requests)
greeting = run(thread,"请只回复准备就绪。不要调用任何 Skill 或工具。")
catalog_request = lab.request_text(catalog_start)


## 2. 显式启用 Skill，暂不读取参考｜必读必跑

用 `SkillInput` 指定文件，观察原生正文加载。任务只要求说明方法，还不需要阈值，所以参考文件应留在磁盘上。


In [ ]:
body_start = len(lab.requests)
method = run(thread,[SkillInput(name="pool-check",path=str(skill_path)),
    TextInput("按这项 Skill 简述核对方法，以及需要阈值时应读哪份文件。此时不需要阈值，不调用工具。")])
body_request = lab.request_text(body_start)


## 3. 新问题出现，再读取参考｜必读必跑

这次需要按具体阈值判断。路径来自 Skill；参考正文要等工具读取后才会进入后续请求。编号只用于检查，不要求回答逐字引用。


In [ ]:
reference_start = len(lab.requests)
decision = run(thread,
    "现在按教学阈值核对：最近 5 分钟连接占用 95%，连接等待 1200ms。"
    "请读取 Skill 指定的参考文件，判断是否需要进一步检查。不要修改文件。")
reference_calls = lab.requests[reference_start:]
reference_first = json.dumps(reference_calls[0],ensure_ascii=False)
reference_last = json.dumps(reference_calls[-1],ensure_ascii=False)


## 4. 执行脚本：输出进入 Context，源码呢？｜必读必跑

Skill 只给脚本路径与用法。脚本只统计报告字符数，不检查引用完整性，也不评分报告质量。让 Agent 执行报告长度检查，看返回值是否进入下一次请求；再看脚本源码的内部标记有没有出现。如果 Agent 自己打开了源码，表里会如实显示，不能宣称“执行脚本绝不会读源码”。

**继续观察 / 可选**：把参考文件扩展成长手册，最初两个阶段的请求会因此变长吗？文件名出现在 Skill 中，与全文已经进入 Context，是同一回事吗？读完表后执行清理。


In [ ]:
script_start = len(lab.requests)
script_result = run(thread,
    f"用 Skill 中的报告检查脚本检查 Agent 工作目录中的报告，报告绝对路径为 {report_path}。"
    "按 Skill 给出的命令执行，把报告绝对路径作为第一个参数，并报告执行结果。"
    "正常执行即可，不要预先阅读脚本源码，不修改文件。")
script_calls = lab.requests[script_start:]
script_first = json.dumps(script_calls[0],ensure_ascii=False)
script_last = json.dumps(script_calls[-1],ensure_ascii=False)
rows = []
for stage,text in [("普通问候",catalog_request),("加载 Skill 正文",body_request),
                   ("请求参考，尚未返回",reference_first),("参考读取后",reference_last),
                   ("执行脚本前",script_first),("执行脚本后",script_last)]:
    rows.append((stage,*["已进入" if marker in text else "未见" for marker in (meta_tag,body_tag,ref_tag,script_tag,output_tag)]))
table(["阶段","描述","正文","参考内容","脚本源码标记","脚本结果标记"],rows)
table(["运行", "实际回答"],[("阈值判断",decision.final_response),("脚本执行",script_result.final_response)])
details("扩展：参考与脚本的真实回执",json.dumps(command_receipts(reference_calls+script_calls),ensure_ascii=False,indent=2))


In [ ]:
# 所有继续观察完成后关闭。
lab.close()
